# Notebook 02 — S&P 500 Data Validation

This notebook validates the persisted S&P 500 master dataset produced by Notebook 01.

**Input:**
`data/raw/sp500_1950_present.csv`

**Purpose:**
- Validate schema and data types
- Validate dates and uniqueness
- Detect duplicate observations
- Detect missing values
- Validate OHLC relationships
- Detect invalid/non-positive prices
- Validate volume
- Detect suspicious gaps
- Compare historical and Yahoo-era data quality
- Produce a machine-readable validation report
- Preserve the raw master dataset by never silently modifying it

**Important:** This notebook does not fabricate, repair, or silently delete observations. Any integrity problem is reported explicitly for investigation.


## 1. Imports

In [ ]:
from pathlib import Path
from datetime import datetime
import json
import os

import numpy as np
import pandas as pd
import matplotlib.pyplot as plt

print("Imports loaded successfully.")


## 2. Project Paths and Configuration

The notebook is independently runnable and does not depend on variables left in Notebook 01's kernel.


In [ ]:
PROJECT_ROOT = Path.cwd()

if PROJECT_ROOT.name == "notebooks":
    PROJECT_ROOT = PROJECT_ROOT.parent

if not (PROJECT_ROOT / "data").exists():
    candidates = [
        Path.cwd(),
        Path.cwd().parent,
        Path("/mnt/data/quant-trading-research"),
    ]
    for candidate in candidates:
        if (candidate / "data").exists() and (candidate / "notebooks").exists():
            PROJECT_ROOT = candidate
            break

RAW_DIR = PROJECT_ROOT / "data" / "raw"
REPORT_DIR = PROJECT_ROOT / "reports" / "generated"
TABLE_DIR = PROJECT_ROOT / "reports" / "tables"

MASTER_PATH = RAW_DIR / "sp500_1950_present.csv"
VALIDATION_REPORT_PATH = REPORT_DIR / "sp500_data_validation_report.json"
INVALID_ROWS_PATH = TABLE_DIR / "sp500_invalid_rows.csv"
MISSING_DATES_PATH = TABLE_DIR / "sp500_missing_weekdays.csv"

EXPECTED_COLUMNS = [
    "Date",
    "Open",
    "High",
    "Low",
    "Close",
    "Adj.Close",
    "Volume",
]

NUMERIC_COLUMNS = [
    "Open",
    "High",
    "Low",
    "Close",
    "Adj.Close",
    "Volume",
]

HISTORICAL_CUTOFF = pd.Timestamp("2018-07-12")
EXPECTED_START = pd.Timestamp("1950-01-03")

REPORT_DIR.mkdir(parents=True, exist_ok=True)
TABLE_DIR.mkdir(parents=True, exist_ok=True)

print(f"Project root: {PROJECT_ROOT}")
print(f"Master dataset: {MASTER_PATH}")
print(f"Validation report: {VALIDATION_REPORT_PATH}")


## 3. Load the Master Dataset

Notebook 02 reads the persisted CSV from disk rather than relying on Notebook 01's in-memory variables.

This makes the notebook independently reproducible.


In [ ]:
if not MASTER_PATH.exists():
    raise FileNotFoundError(
        f"Master dataset not found: {MASTER_PATH}\n"
        "Run Notebook 01 successfully enough to create the master CSV first."
    )

df = pd.read_csv(MASTER_PATH, low_memory=False)

print(f"Rows loaded: {len(df):,}")
print(f"Columns loaded: {list(df.columns)}")
display(df.head())


## 4. Schema Validation

The master CSV must contain exactly:

`Date, Open, High, Low, Close, Adj.Close, Volume`

Unexpected columns are reported rather than silently discarded.


In [ ]:
schema_report = {
    "expected_columns": EXPECTED_COLUMNS,
    "actual_columns": list(df.columns),
    "missing_columns": [c for c in EXPECTED_COLUMNS if c not in df.columns],
    "unexpected_columns": [c for c in df.columns if c not in EXPECTED_COLUMNS],
    "is_multiindex": isinstance(df.columns, pd.MultiIndex),
}

schema_report["exact_column_match"] = (
    list(df.columns) == EXPECTED_COLUMNS
    and not schema_report["is_multiindex"]
)

print(json.dumps(schema_report, indent=2, default=str))

if not schema_report["exact_column_match"]:
    print("\nSCHEMA STATUS: FAIL")
else:
    print("\nSCHEMA STATUS: PASS")


## 5. Date Normalization for Validation

Dates are parsed with `errors="coerce"`.

This validation notebook does not overwrite the source CSV. The normalized copy is used only for analysis.


In [ ]:
validation_df = df.copy()

validation_df["Date"] = pd.to_datetime(
    validation_df["Date"],
    errors="coerce"
)

if getattr(validation_df["Date"].dt, "tz", None) is not None:
    validation_df["Date"] = validation_df["Date"].dt.tz_localize(None)

for column in NUMERIC_COLUMNS:
    if column in validation_df.columns:
        validation_df[column] = pd.to_numeric(
            validation_df[column],
            errors="coerce"
        )

print(validation_df.dtypes)


## 6. Date Integrity Checks

In [ ]:
date_checks = {
    "missing_or_invalid_dates": int(validation_df["Date"].isna().sum()),
    "duplicate_dates": int(validation_df["Date"].duplicated().sum()),
    "dates_sorted": bool(validation_df["Date"].is_monotonic_increasing),
    "minimum_date": (
        validation_df["Date"].min().strftime("%Y-%m-%d")
        if validation_df["Date"].notna().any()
        else None
    ),
    "maximum_date": (
        validation_df["Date"].max().strftime("%Y-%m-%d")
        if validation_df["Date"].notna().any()
        else None
    ),
}

print(json.dumps(date_checks, indent=2))


## 7. Numeric-Type and Missing-Value Checks

In [ ]:
numeric_checks = {}

for column in NUMERIC_COLUMNS:
    if column not in validation_df.columns:
        numeric_checks[column] = {
            "present": False,
            "missing": None,
            "dtype": None,
        }
        continue

    numeric_checks[column] = {
        "present": True,
        "missing": int(validation_df[column].isna().sum()),
        "dtype": str(validation_df[column].dtype),
        "numeric": bool(pd.api.types.is_numeric_dtype(validation_df[column])),
    }

print(json.dumps(numeric_checks, indent=2))


## 8. OHLC Relationship Validation

For every valid OHLC observation:

- `High >= Low`
- `Open >= Low`
- `Open <= High`
- `Close >= Low`
- `Close <= High`

These checks are deliberately strict.

The notebook does **not** drop invalid rows or alter prices.


In [ ]:
ohlc_masks = {
    "High < Low": validation_df["High"] < validation_df["Low"],
    "Open < Low": validation_df["Open"] < validation_df["Low"],
    "Open > High": validation_df["Open"] > validation_df["High"],
    "Close < Low": validation_df["Close"] < validation_df["Low"],
    "Close > High": validation_df["Close"] > validation_df["High"],
}

ohlc_counts = {
    name: int(mask.fillna(False).sum())
    for name, mask in ohlc_masks.items()
}

print(json.dumps(ohlc_counts, indent=2))

ohlc_invalid_mask = pd.Series(False, index=validation_df.index)

for mask in ohlc_masks.values():
    ohlc_invalid_mask |= mask.fillna(False)

print(f"Rows violating at least one OHLC relationship: {int(ohlc_invalid_mask.sum()):,}")


## 9. Identify Invalid OHLC Rows

When invalid rows exist, display them for investigation and save them separately.

The original master CSV is never modified.


In [ ]:
invalid_ohlc_rows = validation_df.loc[
    ohlc_invalid_mask,
    EXPECTED_COLUMNS
].copy()

if len(invalid_ohlc_rows):
    invalid_ohlc_rows.to_csv(
        INVALID_ROWS_PATH,
        index=False,
        date_format="%Y-%m-%d"
    )

    print(f"Invalid OHLC rows: {len(invalid_ohlc_rows):,}")
    print(f"Saved diagnostic table: {INVALID_ROWS_PATH}")
    display(invalid_ohlc_rows.head(25))
else:
    print("No invalid OHLC rows detected.")


## 10. Non-Positive Price Validation

In [ ]:
price_columns = ["Open", "High", "Low", "Close", "Adj.Close"]

non_positive_price_counts = {
    column: int((validation_df[column] <= 0).fillna(False).sum())
    for column in price_columns
}

print(json.dumps(non_positive_price_counts, indent=2))

non_positive_price_mask = (
    validation_df[price_columns]
    .le(0)
    .any(axis=1)
    .fillna(False)
)

print(
    f"Rows containing at least one non-positive price: "
    f"{int(non_positive_price_mask.sum()):,}"
)


## 11. Volume Validation

In [ ]:
negative_volume_count = int(
    (validation_df["Volume"] < 0).fillna(False).sum()
)

missing_volume_count = int(validation_df["Volume"].isna().sum())

print(f"Missing Volume: {missing_volume_count:,}")
print(f"Negative Volume: {negative_volume_count:,}")


## 12. Date Range and Historical Cutoff Validation

The historical foundation is expected to begin at `1950-01-03` and the OpenIntro portion ends at `2018-07-12`.

Notebook 02 reports deviations rather than silently correcting them.


In [ ]:
range_checks = {
    "expected_start": EXPECTED_START.strftime("%Y-%m-%d"),
    "actual_start": (
        validation_df["Date"].min().strftime("%Y-%m-%d")
        if validation_df["Date"].notna().any()
        else None
    ),
    "historical_cutoff": HISTORICAL_CUTOFF.strftime("%Y-%m-%d"),
    "actual_end": (
        validation_df["Date"].max().strftime("%Y-%m-%d")
        if validation_df["Date"].notna().any()
        else None
    ),
    "starts_at_expected_date": (
        validation_df["Date"].min() == EXPECTED_START
        if validation_df["Date"].notna().any()
        else False
    ),
}

print(json.dumps(range_checks, indent=2))


## 13. Historical vs Yahoo-Era Coverage

The dataset is split at the OpenIntro cutoff for diagnostics.

This does not assume that every date in either period must be present; it only establishes where each source era begins/ends.


In [ ]:
historical_part = validation_df[
    validation_df["Date"] <= HISTORICAL_CUTOFF
].copy()

yahoo_part = validation_df[
    validation_df["Date"] > HISTORICAL_CUTOFF
].copy()

coverage_report = {
    "historical_rows": int(len(historical_part)),
    "historical_start": (
        historical_part["Date"].min().strftime("%Y-%m-%d")
        if len(historical_part) else None
    ),
    "historical_end": (
        historical_part["Date"].max().strftime("%Y-%m-%d")
        if len(historical_part) else None
    ),
    "yahoo_rows": int(len(yahoo_part)),
    "yahoo_start": (
        yahoo_part["Date"].min().strftime("%Y-%m-%d")
        if len(yahoo_part) else None
    ),
    "yahoo_end": (
        yahoo_part["Date"].max().strftime("%Y-%m-%d")
        if len(yahoo_part) else None
    ),
}

print(json.dumps(coverage_report, indent=2))


## 14. Missing Weekday Detection

This identifies weekday dates absent from the master dataset.

It does **not** assume every weekday was a trading day because market holidays exist. Missing dates are therefore diagnostic candidates, not automatically errors.

No data is fabricated.


In [ ]:
valid_dates = validation_df["Date"].dropna().dt.normalize()

if len(valid_dates):
    calendar_days = pd.date_range(
        start=valid_dates.min(),
        end=valid_dates.max(),
        freq="D"
    )

    weekdays = calendar_days[calendar_days.dayofweek < 5]
    observed_dates = pd.DatetimeIndex(valid_dates.unique())
    missing_weekdays = weekdays.difference(observed_dates)
else:
    missing_weekdays = pd.DatetimeIndex([])

missing_dates_df = pd.DataFrame({
    "Date": missing_weekdays
})

missing_dates_df.to_csv(
    MISSING_DATES_PATH,
    index=False,
    date_format="%Y-%m-%d"
)

print(f"Missing weekdays identified: {len(missing_dates_df):,}")
print(f"Diagnostic file: {MISSING_DATES_PATH}")

display(missing_dates_df.head(25))


## 15. Inspect Large Date Gaps

Large gaps are especially useful for identifying acquisition or source problems.

A gap is measured between consecutive stored observations.


In [ ]:
sorted_dates = (
    validation_df["Date"]
    .dropna()
    .drop_duplicates()
    .sort_values()
    .reset_index(drop=True)
)

date_gap_df = pd.DataFrame({
    "previous_date": sorted_dates.shift(1),
    "current_date": sorted_dates,
})

date_gap_df["calendar_gap_days"] = (
    date_gap_df["current_date"] - date_gap_df["previous_date"]
).dt.days

large_gaps = (
    date_gap_df[
        date_gap_df["calendar_gap_days"] > 10
    ]
    .dropna()
    .sort_values("calendar_gap_days", ascending=False)
    .reset_index(drop=True)
)

print(f"Gaps greater than 10 calendar days: {len(large_gaps):,}")
display(large_gaps.head(25))


## 16. Duplicate Observation Inspection

In [ ]:
duplicate_mask = validation_df["Date"].duplicated(keep=False)

duplicate_rows = (
    validation_df.loc[duplicate_mask, EXPECTED_COLUMNS]
    .sort_values("Date")
)

print(f"Rows involved in duplicate dates: {len(duplicate_rows):,}")

if len(duplicate_rows):
    display(duplicate_rows.head(50))
else:
    print("No duplicate-date observations detected.")


## 17. Missing-Value Matrix

In [ ]:
missing_summary = (
    validation_df[EXPECTED_COLUMNS]
    .isna()
    .sum()
    .rename("missing_count")
    .to_frame()
)

missing_summary["missing_percent"] = (
    missing_summary["missing_count"] / len(validation_df) * 100
)

display(missing_summary)


## 18. Descriptive Statistics

Descriptive statistics are used only for validation and anomaly inspection.


In [ ]:
display(
    validation_df[NUMERIC_COLUMNS].describe().T
)


## 19. Price Anomaly Scan

This identifies extreme one-day close-to-close changes for investigation.

It does not classify them as errors automatically because genuine market events can produce large moves.


In [ ]:
validation_df = validation_df.sort_values("Date").reset_index(drop=True)

validation_df["daily_close_return"] = (
    validation_df["Close"].pct_change()
)

extreme_moves = (
    validation_df.loc[
        validation_df["daily_close_return"].abs() >= 0.20,
        ["Date", "Close", "daily_close_return"]
    ]
    .sort_values("daily_close_return", key=lambda s: s.abs(), ascending=False)
)

print(f"Observations with absolute close return >= 20%: {len(extreme_moves):,}")
display(extreme_moves.head(30))


## 20. OHLC Consistency Diagnostic Plot

A simple diagnostic plot is generated only when data exists.

It is intended to help visually inspect the price series and does not modify the dataset.


In [ ]:
if len(validation_df):
    fig = plt.figure(figsize=(14, 6))
    plt.plot(validation_df["Date"], validation_df["Close"])
    plt.title("S&P 500 Close Price — Data Validation View")
    plt.xlabel("Date")
    plt.ylabel("Close")
    plt.grid(True, alpha=0.25)
    plt.tight_layout()

    figure_path = PROJECT_ROOT / "reports" / "figures" / "sp500_validation_close.png"
    figure_path.parent.mkdir(parents=True, exist_ok=True)
    fig.savefig(figure_path, dpi=150, bbox_inches="tight")
    plt.show()

    print(f"Figure saved: {figure_path}")


## 21. Build the Complete Validation Report

In [ ]:
missing_counts = {
    column: int(validation_df[column].isna().sum())
    for column in EXPECTED_COLUMNS
    if column in validation_df.columns
}

report = {
    "dataset": {
        "path": str(MASTER_PATH),
        "rows": int(len(validation_df)),
        "columns": list(validation_df.columns),
    },
    "schema": schema_report,
    "date_integrity": date_checks,
    "coverage": coverage_report,
    "missing_values": missing_counts,
    "ohlc_relationships": ohlc_counts,
    "non_positive_prices": non_positive_price_counts,
    "negative_volume": negative_volume_count,
    "missing_weekdays": int(len(missing_dates_df)),
    "large_gaps_over_10_days": int(len(large_gaps)),
    "duplicate_rows": int(len(duplicate_rows)),
    "extreme_close_moves_over_20_percent": int(len(extreme_moves)),
}

VALIDATION_REPORT_PATH.write_text(
    json.dumps(report, indent=2, default=str),
    encoding="utf-8"
)

print(f"Validation report written to: {VALIDATION_REPORT_PATH}")


## 22. Determine Validation Status

A dataset is **VALID** only when the structural and hard-integrity checks pass.

Suspicious returns and calendar gaps are reported separately because they may represent legitimate market behavior or holidays.

Invalid OHLC relationships, duplicate dates, missing required fields, and invalid prices are hard failures.


In [ ]:
hard_failures = {
    "schema": not schema_report["exact_column_match"],
    "missing_or_invalid_dates": date_checks["missing_or_invalid_dates"] > 0,
    "duplicate_dates": date_checks["duplicate_dates"] > 0,
    "dates_not_sorted": not date_checks["dates_sorted"],
    "missing_required_values": any(
        missing_counts.get(column, 1) > 0
        for column in NUMERIC_COLUMNS
    ),
    "ohlc_relationship_violations": any(
        count > 0 for count in ohlc_counts.values()
    ),
    "non_positive_prices": int(non_positive_price_mask.sum()) > 0,
    "negative_volume": negative_volume_count > 0,
    "unexpected_start": not range_checks["starts_at_expected_date"],
}

validation_pass = not any(hard_failures.values())

print("=" * 72)
print("NOTEBOOK 02 VALIDATION STATUS")
print("=" * 72)

for name, failed in hard_failures.items():
    print(f"{name}: {'FAIL' if failed else 'PASS'}")

print()
print(f"OVERALL DATA VALIDATION: {'PASS' if validation_pass else 'FAIL'}")

if not validation_pass:
    print(
        "\nThe master dataset contains one or more hard-integrity failures. "
        "Do not silently repair or delete these observations."
    )


## 23. Final Dataset Preview

In [ ]:
print("FIRST 10 ROWS")
display(validation_df[EXPECTED_COLUMNS].head(10))

print("\nLAST 10 ROWS")
display(validation_df[EXPECTED_COLUMNS].tail(10))

print("\nFINAL SHAPE")
print(validation_df[EXPECTED_COLUMNS].shape)

print("\nFINAL DATE RANGE")
print(
    validation_df["Date"].min().date(),
    "→",
    validation_df["Date"].max().date()
)


## 24. Final Assertions

These assertions are intentionally strict.

If Notebook 01 produced invalid OHLC relationships, this notebook will stop with a clear error rather than pretending the dataset is ready for downstream research.


In [ ]:
assert schema_report["exact_column_match"], (
    "Schema validation failed. Inspect the schema report."
)

assert date_checks["missing_or_invalid_dates"] == 0, (
    "Invalid or missing dates detected."
)

assert date_checks["duplicate_dates"] == 0, (
    "Duplicate dates detected."
)

assert date_checks["dates_sorted"], (
    "Dates are not chronologically sorted."
)

assert all(
    missing_counts.get(column, 1) == 0
    for column in NUMERIC_COLUMNS
), "Missing required OHLCV values detected."

assert all(
    count == 0 for count in ohlc_counts.values()
), "Invalid OHLC relationships detected."

assert int(non_positive_price_mask.sum()) == 0, (
    "Non-positive prices detected."
)

assert negative_volume_count == 0, (
    "Negative volume detected."
)

assert range_checks["starts_at_expected_date"], (
    "Dataset does not begin at the expected OpenIntro date."
)

print("All hard validation assertions PASSED.")


# Notebook 02 Complete

Notebook 02 is the independent quality gate between data acquisition and downstream quantitative research.

It does not modify the master dataset.

If validation fails, investigate Notebook 01/source data before continuing to exploratory analysis.

**Required workflow:**

1. Run Notebook 01.
2. Confirm the master CSV exists.
3. Run this Notebook 02 from top to bottom.
4. Review the validation report and diagnostic tables.
5. Only after the data-quality issues are resolved should Notebook 03 begin.
